In [1]:
import numpy as np
import math
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import time

In [3]:
#define the shape of the environment (i.e., its states). States for each node are 0,1,2,...,m_star where
# 0 = inactive memory, 1 to m_star = active memory
m_star = 1  #number of states = m_star + 1
n_node = 5  #number of nodes
state = np.zeros([n_node, n_node])
n_state = (m_star+1)**(int(n_node*(n_node-1)/2))
n_action = (2)**(2*n_node - 1)
# 0 = wait for photon/BM, 1 = request for photon/BM (on nodes we do BM, on physical links entangled pair request)
action = np.zeros([n_node, n_node])

q_values = np.zeros([n_state,n_action])

In [4]:
#check if is an allowed state
def is_allowed_state(state):
    allowed = True

    for i in range(n_node):
        if state[i][i]>0:
            allowed = False
            break
            
        c1 = 0
        for j in range(i,n_node):
            if state[i][j]>0:
                c1+=1
        if c1>1:
            allowed = False
            break
            
    if allowed:
        for j in range(n_node):
            c2 = 0
            for i in range(j):
                if state[i][j]>0:
                    c2+=1
            if c2>1:
                allowed = False
                break
                
    if allowed:
        for i in range(n_node):
            for j in range(i+2, n_node):
                if state[i][j] > 0 and allowed:
                    if state[i][i+1]>0 or state[i][j-1]:
                        allowed = False
                    
                    for k in range(i+1,j):
                        if allowed==False:
                            break
                        for l in range(j+1,n_node):
                            if state[k][l]>0:
                                allowed=False
                                break
                        
                if not allowed:
                    break
            if not allowed:
                break
                
    return(allowed)   

# check if allowed action
def is_allowed_action(action):
    allowed = True
    
    if action[0][0] == 1 or action[n_node-1][n_node-1] == 1:
        return False
    
    for i in range(1,n_node-1):
            if action[i][i]>0:
                if action[i-1][i]==1 or action[i][i+1]==1:
                    allowed = False
                    break
                    
    if allowed:
        for i in range(n_node):
            for j in range(i+2,n_node):
                if action[i][j]>0:
                    allowed = False
                    break
            if not allowed:
                break
    
    return allowed                           

In [5]:
# Reward function
def rewards(state_index):
    state = get_state_from_index(state_index)
    if is_allowed_state(state):
        if is_terminal_state(state):
            reward = 100#/state[0][n_node-1]  #remove comment for fidelity based rewards
        else:
            reward = -1            
    else:
        reward = -1000
    return reward

In [6]:
# Train the Model
# Our next task is for our AI agent to learn about its environment by implementing a Q-learning model. 
# The learning process will follow these steps:
# Choose a random, non-terminal state for the agent to begin this new episode.
# Choose an action for the current state. Actions will be chosen using an epsilon greedy algorithm. 
# This algorithm will usually choose the most promising action for the AI agent, 
# but it will occasionally choose a less promising option in order to encourage the agent to explore the environment.
# Perform the chosen action, and transition to the next state (i.e., move to the next location).
# Receive the reward for moving to the new state, and calculate the temporal difference.
# Update the Q-value for the previous state and action pair.
# If the new (current) state is a terminal state, go to #1. Else, go to #2.
# This entire process will be repeated across 1000 episodes. 
# This will provide the AI agent sufficient opportunity to learn the shortest paths 
# Define Helper Functions


#define a function that determines if the specified location is a terminal state
def is_terminal_state(current_state):
  #if the state has a link between 1st and last node then it is terminal
    if current_state[0][n_node-1] > 0 and is_allowed_state(current_state):  
        return True
    else:
        return False
    
def get_state_index(state):
    index = 0
    n_max = int(n_node*(n_node-1)/2) #total number of links (physical + virtual)
    n = 0
    for i in range(n_node):
        for j in range(i+1,n_node):
            n = n+1
            index = index + state[i][j]*(m_star+1)**(n_max - n)
    return int(index)

def get_state_from_index(index):
    base = m_star+1
    base_num = ""
    while index>0:
        dig = int(index%base)
        base_num += str(dig)
        index //= base
    base_num = base_num[::-1]
    a = np.zeros([int(n_node*(n_node-1)/2)])
    pad = int(n_node*(n_node-1)/2)-len(base_num)
    for k in range(int(n_node*(n_node-1)/2)):
        if k < pad:
            a[k] = 0
        else:
            a[k] = int(base_num[k-pad])
            
    state = np.zeros([n_node,n_node])
    k = 0
    for i in range(n_node):
        for j in range(i+1,n_node):
            k = k + 1
            state[i][j] = a[k-1]
            state[j][i] = state[i][j]
            
    return state

def get_action_index(state):
    index = 0
    n_max = 2*n_node - 1 #total number of physical links + number of nodes
    n = 0
    for i in range(n_node):
        for j in range(i,i+2):
            if j<n_node:
                n = n+1
                index = index + state[i][j]*(2)**(n_max - n)
    return int(index)

def get_action_from_index(index):
    base = 2
    base_num = ""
    while index>0:
        dig = int(index%base)
        base_num += str(dig)
        index //= base
    base_num = base_num[::-1]
    
    a = np.zeros([2*n_node - 1])
    pad = 2*n_node - 1 - len(base_num)
    for k in range(2*n_node - 1):
        if k < pad:
            a[k] = 0
        else:
            a[k] = int(base_num[k-pad])
            
    action = np.zeros([n_node,n_node])
    k = 0
    for i in range(n_node):
        for j in range(i,i+2):
            if j<n_node:
                k = k + 1
                action[i][j] = a[k-1]
                action[j][i] = action[i][j]
            
    return action

ct=0
cta=0
allowed_states = []
allowed_actions = []
for i in range(n_state):
    if is_allowed_state(get_state_from_index(i)):
        ct+=1
        allowed_states.append(i)       

for i in range(n_action):
    if is_allowed_action(get_action_from_index(i)):
        cta+=1
        allowed_actions.append(i)       
allowed_states = np.array(allowed_states)
allowed_actions = np.array(allowed_actions)

#q_values = np.zeros([ct,cta])
print([ct,cta])
def get_random_state():
    i = np.random.randint(0,ct)
    return get_state_from_index(allowed_states[i])

def get_random_action():
    i = np.random.randint(0,cta)
    return get_action_from_index(allowed_actions[i])

#define a function that will choose a random, non-terminal starting state
def get_starting_state():
    
#   get a random state
    current_state = get_random_state()

#   continue choosing random state until a non-terminal state is identified
    while is_terminal_state(current_state):
        current_state = get_random_state()

    return current_state


#define an epsilon greedy algorithm that will choose which action to take next
def get_next_action(current_state, epsilon):
  #if a randomly chosen value between 0 and 1 is less than epsilon, 
  #then choose the most promising value from the Q-table for this state.
    state_index = get_state_index(current_state)
    
    if np.random.random() < epsilon:
        action_index = np.argmax(q_values[state_index])
        opt_action = get_action_from_index(action_index)
        #return opt_action
        if is_allowed_action(opt_action): 
            return opt_action
        else:
            return get_random_action()
    else: #choose a random action
        return get_random_action()

#define a function that will get the next location based on the chosen action
def get_next_state(current_state, action, p_l, p_bm):
    
    new_state = np.zeros([n_node,n_node])
    
    for i in range(n_node):
        for j in range(n_node):
            new_state[i][j] = current_state[i][j]
    

    for i in range(n_node-1):
        j=i+1
        if action[i][j] == 1: #request entanglement
            if np.random.random()<=p_l[i][j]:
                new_state[i][j] = 1
            else:
                new_state[i][j] = 0        

        new_state[j][i] = new_state[i][j]   
    
    
    #Bell measurements

    for i in range(1,n_node-1):

        if action[i][i] == 1:  # request BM
            flag1 = 0; flag2 = 0;
            for k in range(i+1,n_node):
                if new_state[i][k] > 0:
                    node1 = k
                    node1_val = new_state[i][k]
                    flag1 = 1
                    break
            for l in range(i):
                if new_state[l][i] > 0:
                    node2 = l
                    node2_val = new_state[l][i]
                    flag2 = 1
                    break

            if flag1==1 and flag2==1:
                if np.random.random()<p_bm: #BM success
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = node1_val + node2_val - 1 #max(node1_val,node2_val) #
#                     print(new_state[node2][node1])
                    if new_state[node2][node1] > m_star:
#                         print('hellloooo')
                        new_state[node2][node1] = 0
                else:                       #BM failure
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = 0
            else:
                if flag1==1 and flag2==0:
                    new_state[i][node1] = 0
                if flag2==1 and flag1==0:
                    new_state[node2][i] = 0
                    
        for i in range(n_node):
            for j in range(i+1,n_node):
                new_state[j][i] = new_state[i][j]
                
    flagg=0
    for i in range(n_node-1):
        if action[i][i+1]>0:
            flagg=1
            break
            
    if flagg==1:
    # wait               
        for i in range(n_node-1):
            for j in range(i+1,n_node):
                if action[i][j] == 0: #wait
                    if new_state[i][j] > 0:
                        new_state[i][j] = (new_state[i][j] + 1)%(m_star+1)
                        new_state[j][i] = new_state[i][j]
                        
    return new_state

[42, 34]


In [14]:
for i in allowed_actions:
    if get_action_from_index(i)[1][2]==1 and get_action_from_index(i)[2][3]==1:
        print(get_action_from_index(i))
        print("")


[[0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 1. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0.]]

[[0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 1. 0. 1. 0.]
 [0. 0. 1. 0. 1.]
 [0. 0. 0. 1. 0.]]

[[0. 1. 0. 0. 0.]
 [1. 0. 1. 0. 0.]
 [0. 1. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0.]]

[[0. 1. 0. 0. 0.]
 [1. 0. 1. 0. 0.]
 [0. 1. 0. 1. 0.]
 [0. 0. 1. 0. 1.]
 [0. 0. 0. 1. 0.]]



In [15]:
(4/34 - 100/34/34)/(10/34)/(24/34)

0.14999999999999994

In [16]:
q_values = np.random.random([n_state, n_action])
c1 = []
c=[]
c2=[]
for i in range(10000):
#     action = get_next_action(get_random_state(), 0)
    action=get_random_action()
    c.append(action[1][2]*action[2][3])
    c1.append(action[1][2])
    c2.append(action[2][3])
    
print(np.mean(c1))
print(np.mean(c2))   
print(np.mean(c))
print(np.std(c1))
print(np.std(c2))
print((np.mean(c)-np.mean(c1)*np.mean(c2))/np.std(c1)/np.std(c2))

0.2942
0.2988
0.1172
0.4556823016093559
0.4577319739760377
0.14044006866816836


In [34]:
def training_network(epsilon, discount_factor, learning_rate, p_l, p_bm, total_episodes):
    
    #run through 500 training episodes
    for episode in range(int(total_episodes)):
        if episode%5000==0:
            print(episode)
#             if episode>150000 and epsilon<0.9:
#                 epsilon+=0.05
        
        #get the starting state for this episode
        state = get_starting_state()
        
        #continue taking actions until we reach a terminal state or we do 1000 actions
        trial_num = 0
        while trial_num<200 and is_terminal_state(state)==False:
            
            #choose which action to take
            action = get_next_action(state, epsilon)
            
            #perform the chosen action, and transition to the next state
            old_state = np.zeros([n_node,n_node]) #store the old state
            for i in range(n_node):
                for j in range(n_node):
                    old_state[i][j] = state[i][j]
                    
            state = get_next_state(old_state, action, p_l, p_bm)
            
            old_state_index = get_state_index(old_state)
            state_index = get_state_index(state)
            action_index = get_action_index(action)
            
            #receive the reward for moving to the new state, and calculate the temporal difference
            reward = rewards(state_index)
            old_q_value = q_values[old_state_index][action_index]
            temporal_difference = reward + (discount_factor * np.max(q_values[state_index])) - old_q_value

            #update the Q-value for the previous state and action pair
            new_q_value = old_q_value + (learning_rate * temporal_difference)
            q_values[old_state_index][action_index] = new_q_value
            trial_num = trial_num + 1
        
    print('Training complete!')

In [35]:
#Define a function that will get the shortest path between any initial state and final state.
        
def evolve_state(start_state, p_l, p_bm, steps, printing=False):
    trial_num = 0
    #if this is a 'legal' starting state
    current_state = np.array([i for i in start_state])
    evolution_path = []
    evolution_path.append(current_state)
    #continue moving along the path until we reach the goal (i.e., all active node state) or 5000 actions
    while trial_num<steps:
        #get the best action to take
        action = get_next_action(current_state, 1.)
        if printing:
            print(np.array(action))
            print("")
        
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm)])
        if printing:
            print(np.array(current_state))
            print("")
        evolution_path.append(current_state)
        trial_num = trial_num + 1
    return evolution_path

#Define a function that will get the shortest path between any initial state and final state.
def evolve_action(start_state, p_l, p_bm, steps):
    trial_num = 0
    #if this is a 'legal' starting state
    current_state = np.array([i for i in start_state])
    evolution_path = []
    #continue moving along the path until we reach the goal (i.e., all active node state) or 5000 actions
    while trial_num<steps:
        #get the best action to take
        action = get_next_action(current_state, 1.)
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm)])
        evolution_path.append(action)
        trial_num = trial_num + 1
    return evolution_path

In [36]:
def shortest_path(start_state, p_l, p_bm, cc=False, nreq=False):
    
    current_state = np.array([i for i in start_state])
    evolution_path = []
    evolution_path.append(current_state)
    nr=0
    trial_num=0
    #continue moving along the path until we reach the goal (i.e., all active node state)
    while not is_terminal_state(current_state):
        #get the best action to take
        action = get_next_action(current_state, 1.)
        
        flag=0
        for i in range(n_node-1):
            if action[i][i+1]>0:
                flag=1
                break
        
        if nreq==True:
            for i in range(n_node-1):
                if action[i][i+1]>0:
                    nr+=1
#         print(action)
#         print("")
        
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm)])
#         print(current_state)
#         print("")
        if cc==False:
            if flag==1:
                evolution_path.append(current_state)
        else:
            evolution_path.append(current_state)
        trial_num+=1
    evolution_path.append(current_state)    
    if nreq==False:    
        return evolution_path
    else:
        return evolution_path,nr
    
def shortest_path(start_state, p_l, p_bm, cc=False, nreq=False):
    
    current_state = np.array([i for i in start_state])
    evolution_path = []
    evolution_path.append(current_state)
    nr=0
    trial_num=0
    #continue moving along the path until we reach the goal (i.e., all active node state)
    while not is_terminal_state(current_state):
        #get the best action to take
        action = get_next_action(current_state, 1.)
        
        flag=0
        for i in range(n_node-1):
            if action[i][i+1]>0:
                flag=1
                break
        
        if nreq==True:
            for i in range(n_node-1):
                if action[i][i+1]>0:
                    nr+=1
#         print(action)
#         print("")
        
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm)])
#         print(current_state)
#         print("")
        if cc==False:
            if flag==1:
                evolution_path.append(current_state)
        else:
            evolution_path.append(current_state)
        trial_num+=1
    evolution_path.append(current_state)    
    if nreq==False:    
        return evolution_path
    else:
        return evolution_path,nr
    
def shortest_path_s_a(start_state, p_l, p_bm, cc=False, nreq=False):
    
    current_state = np.array([i for i in start_state])
    evolution_path = []
    action_path = []
    nr=0
    trial_num=0
    #continue moving along the path until we reach the goal (i.e., all active node state)
    while not is_terminal_state(current_state):
        evolution_path.append(current_state)
        #get the best action to take
        action = get_next_action(current_state, 1.)
        action_path.append(action)
        flag=0
        for i in range(n_node-1):
            if action[i][i+1]>0:
                flag=1
                break
        
        if nreq==True:
            for i in range(n_node-1):
                if action[i][i+1]>0:
                    nr+=1
#         print(action)
#         print("")
        
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm)])
#         print(current_state)
#         print("")
#         if cc==False:
#             if flag==1:
#                 evolution_path.append(current_state)
#         else:
#             evolution_path.append(current_state)
        trial_num+=1
#     evolution_path.append(current_state)    
    if nreq==False:    
        return evolution_path,action_path
    else:
        return evolution_path,nr

def evolve_sa(start_state, p_l, p_bm, steps, printing=False):
    trial_num = 0
    #if this is a 'legal' starting state
    current_state = np.array([i for i in start_state])
    evolution_path = []
    action_path = []
#     evolution_path.append(current_state)
    #continue moving along the path until we reach the goal (i.e., all active node state) or 5000 actions
    while trial_num<steps:
        evolution_path.append(current_state)
        #get the best action to take
        action = get_next_action(current_state, 0.)
        action_path.append(action)
        if printing:
            print(np.array(action))
            print("")
        
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state_SA(current_state, action, p_l, p_bm)])
        if printing:
            print(np.array(current_state))
            print("")
#         evolution_path.append(current_state)
        trial_num = trial_num + 1
    return evolution_path,action_path

In [26]:
from scipy.sparse import csc_matrix
total_episodes = 300000
learning_rate = 0.01

p_l = np.zeros([n_node,n_node])
p_l[0][1]=0.5
p_l[1][2]=0.5
p_l[2][3]=0.5
p_l[3][4]=0.5

for p_bm in [1.0]:
    epsilon = 0.15
    discount_factor = 0.8
    # q_values = np.load("q_valuesn5m3_200000_pl0.7_pbm0.5"+"eps_"+str(0.5)+"df_"+str(discount_factor)+".npy",allow_pickle=True)
    # q_values = q_values[()].A
    q_values = np.zeros([n_state,n_action])
    t0=time.time()
    training_network(epsilon, discount_factor, learning_rate, p_l, p_bm, total_episodes)
    print(time.time()-t0)
    np.save("q_valuesn5m1_pauli_law_300000_pbm"+str(p_bm)+".npy",csc_matrix(q_values))


0
5000
10000
15000
20000
25000
30000
35000
40000
45000
50000
55000
60000
65000
70000
75000
80000
90000
95000
100000
105000
110000
115000
120000
125000
130000
135000
140000
145000
150000
155000
160000
165000
170000
175000
180000
185000
190000
195000
200000
205000
210000
215000
220000
225000
230000
235000
240000
245000
250000
255000
260000
265000
270000
275000
280000
285000
290000
295000
Training complete!
8801.680980443954


In [174]:
#average waiting time K_n with/without swap steps (CC='True' or 'False')
n_node = 4
m_star = 3
mean_list=[]
mean_list_fid=[]
std_list = []
std_list_fid = []
p_bm=0.5
for pl in [0.6]:#np.linspace(0.2,1.0,9):
#     q_values = np.load("q_valuesn5m1_pauli_law_300000_pbm"+str(p_bm)+".npy", allow_pickle=True)
#     q_values = q_values[()].A
    q_values = q_values_swap_asap_4
#     if pl<0.1:
#         q_values = np.load("q_valuesn4m3_wtfid_asym_pauli_law_300000_delta_pl"+str(0.2)+"_pbm0.5.npy", allow_pickle=True)
#         q_values = q_values[()].A
#     if pl>0.1 and pl<0.2:
#         q_values = np.load("q_valuesn4m3_wtfid_asym_pauli_law_300000_delta_pl"+str(0.2)+"_pbm0.5.npy", allow_pickle=True)
#         q_values = q_values[()].A
#     if pl>0.2 and pl<0.3:
#         q_values = np.load("q_valuesn4m3_wtfid_asym_pauli_law_300000_delta_pl"+str(0.3)+"_pbm0.5.npy", allow_pickle=True)
#         q_values = q_values[()].A
#     if pl==0.3:
#         q_values = np.load("q_valuesn4m3_wtfid_asym_pauli_law_300000_delta_pl"+str(0.3)+"_pbm0.5.npy", allow_pickle=True)
#         q_values = q_values[()].A
    
    
    p_l = np.zeros([n_node,n_node])
    p_l[0][1] = pl
    p_l[1][2] = pl
    p_l[2][3] = pl
#     p_l[3][4] = pl
    

    mean_mean=[]
    mean_fid=[]

    discount_factor = 0.8
    epsilon = 0.15

    for p in range(10):

        print(p)
        K_n = []
        K_n_fid = []
        for k in range(500):
            print(k)
            c = [0]*(n_node-1)
            state = np.zeros([n_node, n_node])
            K_n.append(len(shortest_path(state, p_l, p_bm, cc=False, nreq=False))-2)
            K_n_fid.append(shortest_path(state, p_l, p_bm, cc=False, nreq=False)[-1][0][n_node-1])
            
        mean_mean.append(np.mean(K_n))
        mean_fid.append(np.mean(K_n_fid))
    
    print(np.mean(mean_mean))
    print(np.mean(mean_fid))
    mean_list.append(np.mean(mean_mean))
    mean_list_fid.append(np.mean(mean_fid))
    std_list.append(np.std(mean_mean))
    std_list_fid.append(np.std(mean_fid))

0
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276


267
268
269
270
271
272
273
274
275
276
277
278
279
280
281
282
283
284
285
286
287
288
289
290
291
292
293
294
295
296
297
298
299
300
301
302
303
304
305
306
307
308
309
310
311
312
313
314
315
316
317
318
319
320
321
322
323
324
325
326
327
328
329
330
331
332
333
334
335
336
337
338
339
340
341
342
343
344
345
346
347
348
349
350
351
352
353
354
355
356
357
358
359
360
361
362
363
364
365
366
367
368
369
370
371
372
373
374
375
376
377
378
379
380
381
382
383
384
385
386
387
388
389
390
391
392
393
394
395
396
397
398
399
400
401
402
403
404
405
406
407
408
409
410
411
412
413
414
415
416
417
418
419
420
421
422
423
424
425
426
427
428
429
430
431
432
433
434
435
436
437
438
439
440
441
442
443
444
445
446
447
448
449
450
451
452
453
454
455
456
457
458
459
460
461
462
463
464
465
466
467
468
469
470
471
472
473
474
475
476
477
478
479
480
481
482
483
484
485
486
487
488
489
490
491
492
493
494
495
496
497
498
499
5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
2

32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
278
279
280
281
282
283
284
285
286
287
288
289
290
291
292
293
294
295
296
297
298


In [13]:
mean_list_fid

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

In [24]:
#mean cut-off of links
n_node = 4
m_star = 3
mean_list=[]
mean_age_list=[]
for pl in [0.0]:
    p_l = np.zeros([n_node,n_node])
    p_l[0][1] = 0.3
    p_l[1][2] = 0.6
    p_l[2][3] = 0.9
#     p_l[3][4] = 0.6
    print(p_l)
    mean_mean=[]
    mean_age = np.zeros([n_node-1,10])
    p_bm = 0.5
    discount_factor = 0.8
    epsilon = 0.15
    q_values = np.load("q_valuesn4m3_asym_pauli_law_600000_delta_pl0.3_pbm0.5.npy",allow_pickle=True)
    q_values = q_values[()].A
    for p in range(10):
        print(p)
        K_n = []
        K_n_age = np.zeros([n_node-1,5000])
        for k in range(5000):
            state = np.zeros([n_node, n_node])
            K_n.append(len(shortest_path(state, p_l, p_bm, cc=False, nreq=False))-1)
            c = [0]*(n_node-1)
            states,actions = shortest_path_s_a(state, p_l, p_bm, cc=False, nreq=False)
            l=0
            for state in states:
                action = actions[l]
                l+=1
                for i in range(n_node-1):
                        if state[i][i+1]>0 and action[i][i+1]==1:
                            K_n_age[i][k] = K_n_age[i][k]+state[i][i+1]
                            c[i]+=1

            for i in range(n_node-1):
                if c[i]>0:
                    K_n_age[i][k] = K_n_age[i][k]/c[i]

        for i in range(n_node-1):
            mean_age[i][p] = np.mean(np.array(K_n_age[i])[K_n_age[i]>0])  

        mean_mean.append(np.mean(K_n))
    print(p_l)
    print([np.mean(mean_mean),np.std(mean_mean)])
    print("")
    mean_list.append(np.mean(mean_mean))
    mean_age_list.append([np.mean(mean_age[0]), np.mean(mean_age[1]), np.mean(mean_age[2])])
    print([np.mean(mean_age[0]), np.mean(mean_age[1]), np.mean(mean_age[2])])

[[0.  0.3 0.  0. ]
 [0.  0.  0.6 0. ]
 [0.  0.  0.  0.9]
 [0.  0.  0.  0. ]]
0
1
2
3
4
5
6
7
8
9
[[0.  0.3 0.  0. ]
 [0.  0.  0.6 0. ]
 [0.  0.  0.  0.9]
 [0.  0.  0.  0. ]]
[16.106900000000003, 0.19270533464333559]

[3.0, 2.0, 1.0626089801348395]


In [20]:
def get_next_state_SA(current_state, action, p_l, p_bm):
    
    new_state = np.zeros([n_node,n_node])

    for i in range(n_node):
        for j in range(n_node):
            new_state[i][j] = current_state[i][j]


    for i in range(n_node-1):
        j=i+1
        if action[i][j] == 1: #request entanglement
            if np.random.random()<=p_l[i][j]:
                new_state[i][j] = 1
            else:
                new_state[i][j] = 0        

        new_state[j][i] = new_state[i][j]   


    #Bell measurements

    for i in range(1,n_node-1):

        if action[i][i] == 1:  # request BM
            flag1 = 0; flag2 = 0;
            for k in range(i+1,n_node):
                if new_state[i][k] > 0:
                    node1 = k
                    node1_val = new_state[i][k]
                    flag1 = 1
                    break
            for l in range(i):
                if new_state[l][i] > 0:
                    node2 = l
                    node2_val = new_state[l][i]
                    flag2 = 1
                    break

            if flag1==1 and flag2==1:
                if np.random.random()<p_bm: #BM success
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = node1_val + node2_val - 1  #max(node1_val,node2_val)
                    if new_state[node2][node1] > m_star:
                        new_state[node2][node1] = 0
                else:                       #BM failure
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = 0
#                 else:
#                     if flag1==1 and flag2==0:
#                         new_state[i][node1] = 0
#                     if flag2==1 and flag1==0:
#                         new_state[node2][i] = 0

        for i in range(n_node):
            for j in range(i+1,n_node):
                new_state[j][i] = new_state[i][j]
    flag=0
    for i in range(n_node-1):
        if action[i][i+1]>0:
            flag=1
            break

    if flag==1:
        # wait for all links and request on physical links                
        for i in range(n_node-1):
            for j in range(i+1,n_node):
                if action[i][j] == 0: #wait
                    if new_state[i][j] > 0:
                        new_state[i][j] = (new_state[i][j] + 1)%(m_star+1)
                new_state[j][i] = new_state[i][j]

    return new_state

In [29]:
#unequal time correlation functions
n_node = 5
m_star = 3
mean_list=[]
std_list=[]
for dist in [0]:
    print(dist)
    # steps = 10
    # for p_l in [0.4]:#np.linspace(0.4,0.9,6):
    p_l = np.zeros([n_node,n_node])
    for i in range(n_node-1):
        j=i+1
        if i==1 or i==2:
            p_l[i][j] = 0.6
        else:
            p_l[i][j] = 0.6
    for tau in np.linspace(30,60,5):
        mean_mean=[]
        p_bm = 0.5
        discount_factor = 0.8
        epsilon = 0.15
#         q_values = 100*np.random.rand(n_state,n_action)
#         q_values = np.load("q_valuesn5m3_pauli_law_300000_pl0.5_pbm0.5eps_0.15df_0.8.npy",allow_pickle=True)
#         q_values = q_values[()].A
        q_values = q_values_swap_asap_4
        for p in range(10):
            print(p)
            K_n = []
            K_n1 = []
            K_n2 = []
            for k in range(1000):
                state = np.zeros([n_node, n_node])
                states,actions = evolve_sa(state, p_l, p_bm, tau)
                if states[-1][0+dist][1+dist]>0:
                    states[-1][0+dist][1+dist] = 1
                K_n.append(actions[4][0][1]*states[-1][0+dist][1+dist])
                K_n1.append(actions[4][0][1])
                K_n2.append(states[-1][0+dist][1+dist])

            mean_mean.append(np.abs((np.mean(K_n) - np.mean(K_n1)*np.mean(K_n2))/np.std(K_n1)/np.std(K_n2)))
    #     print(p_l)
        print([np.mean(mean_mean),np.std(mean_mean)])
        print("")
        mean_list.append(np.mean(mean_mean))
        std_list.append(np.std(mean_mean))

0
0
1
2
3
4
5
6
7
8
9
[0.027027638843501865, 0.016888269867910092]

0
1
2
3
4
5
6
7
8
9
[0.039792174203543015, 0.0176842104100219]

0
1
2
3
4
5
6
7
8
9
[0.026301722174994917, 0.020331345469129083]

0
1
2
3
4
5
6
7
8
9
[0.020206995798556927, 0.014635815296007527]

0
1
2
3
4
5
6
7
8
9
[0.029526228397251792, 0.014942176963431643]



In [33]:
#equal time correlation functions  #newwwwwwww newwwwww newwwwww
n_node = 5
m_star = 3
mean_list=[]
std_list=[]
for dist in [1,2,3]:
    p_l = np.zeros([n_node,n_node])
    for i in range(n_node-1):
        j=i+1
        p_l[i][j] = 0.6
        
    for tau in range(5,6):
        mean_mean=[]
        p_bm = 0.5
        discount_factor = 0.8
        epsilon = 0.15
#         q_values = np.random.rand(n_state,n_action)
        q_values = np.load("q_valuesn5m3_pauli_law_300000_pl0.5_pbm0.5eps_0.15df_0.8.npy",allow_pickle=True)
        q_values = q_values[()].A
#         q_values = q_values_swap_asap_4
        for p in range(10):
            print(p)
            K_n = []
            K_n1 = []
            K_n2 = []
            for k in range(1000):
#                 state = np.zeros([n_node, n_node])
                state = get_random_state()
                action = get_next_action(state, 1)
                if state[0+dist][1+dist]>0:
                    state[0+dist][1+dist] = 1
                K_n.append(action[0][1]*action[0+dist][1+dist])
                K_n1.append(action[0][1])
                K_n2.append(action[0+dist][1+dist])

            mean_mean.append(np.abs((np.mean(K_n) - np.mean(K_n1)*np.mean(K_n2))/np.std(K_n1)/np.std(K_n2)))
    #     print(p_l)
        print([np.mean(mean_mean),np.std(mean_mean)])
        print("")
        mean_list.append(np.mean(mean_mean))
        std_list.append(np.std(mean_mean))

0
1
2
3
4
5
6
7
8
9
[0.5869320504286255, 0.0204078820875935]

0
1
2
3
4
5
6
7
8
9
[0.28857707334544475, 0.031673129217885045]

0
1
2
3
4
5
6
7
8
9
[0.28551364772298143, 0.015778345801853465]



In [42]:
def build_action_SWAP_ASAP(state,n_node,m_star):
    
    action = np.zeros([n_node,n_node])

    node_list = []
    for i in range(n_node-1):

        if i==0:
            virt_f=0
            for k in range(i+1,n_node):
                if state[i][k]>0:
                    virt_f=1
                    break
            if virt_f==0 or state[i][i+1]==m_star:
                action[i][i+1] = 1
                action[i+1][i] = 1 

        else:
            flag_inside_virt=0
            for l in range(i):
                for m in range(i+1,n_node):
                    if state[l][m]>0:
                        flag_inside_virt = 1
                        break
                if flag_inside_virt == 1:
                    break

            if flag_inside_virt==0:            
                virt_f=0
                for k in range(i+1,n_node):
                    if state[i][k]>0:
                        virt_f=1
                        break
                if virt_f==0 or state[i][i+1]==m_star:
                    action[i][i+1] = 1
                    action[i+1][i] = 1
            else:
                for x in range(l+1,m-1):
                    if state[x][x+1]==0 or state[x][x+1]==m_star:
                        print('hello')
                        action[x][x+1] = 1
                        action[x+1][x] = 1                 


    for i in range(1,n_node-1):
        f1=0; f2=0;
        for k in range(i+1,n_node):
            if state[i][k] > 0:
                f1=1
                break
        for l in range(i):
            if state[l][i] > 0:
                f2=1
                break

        if f1==1 and f2==1:
            action[i][i] = 1
            action[i-1][i] = 0
            action[i][i-1] = 0
            action[i][i+1] = 0
            action[i+1][i] = 0
        if f1==1 and f2==0:
            if state[i][k]==m_star:
                action[i][i+1] = 1
                action[i+1][i] = 1

    return(action)

q_values_swap_asap_3 = np.zeros([n_state,n_action])

for state_index in allowed_states: 
    state = get_state_from_index(state_index)
    action = build_action_SWAP_ASAP(state,n_node,m_star)
    print("s:")
    print(state)
    print("a:")
    print(action)
    print("")
    print("")
#     action_index = get_action_index(action)
#     q_values_swap_asap_4[state_index][action_index] = 100

s:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
a:
[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]


s:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]]
a:
[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 0.]]


s:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 2.]
 [0. 0. 2. 0.]]
a:
[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 0.]]


s:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 3.]
 [0. 0. 3. 0.]]
a:
[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]


s:
[[0. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 0.]
 [0. 1. 0. 0.]]
a:
[[0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


s:
[[0. 0. 0. 0.]
 [0. 0. 0. 2.]
 [0. 0. 0. 0.]
 [0. 2. 0. 0.]]
a:
[[0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


s:
[[0. 0. 0. 0.]
 [0. 0. 0. 3.]
 [0. 0. 0. 0.]
 [0. 3. 0. 0.]]
a:
[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 0.]]


s:
[[0. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 0.]]
a:
[[0. 1. 0. 0.]
 [1. 0. 

In [22]:
[np.mean(mean_age[0]), np.mean(mean_age[1]), np.mean(mean_age[2]), np.mean(mean_age[3])]

[1.2774350299374801, 1.727836417021623, 1.2181124767180163, 1.223934567041775]

In [12]:
mean_list

[11.747900000000001,
 11.804400000000001,
 12.351,
 12.5823,
 13.4061,
 16.851499999999998,
 22.141700000000004]

In [13]:
mean_list_fid

[1.6785999999999999, 1.6893, 1.6853000000000002, 1.7116, 1.7216, 1.3834, 1.157]

In [35]:
#define a function that will get the next location based on the chosen action
def get_next_state(current_state, action, p_l, p_bm):
    
    new_state = np.zeros([n_node,n_node])
    
    for i in range(n_node):
        for j in range(n_node):
            new_state[i][j] = current_state[i][j]
                
                       
    for i in range(n_node-1):
        j=i+1
        if action[i][j] == 1: #request entanglement
            if np.random.random()<=p_l[i][j]:
                new_state[i][j] = 1
            else:
                new_state[i][j] = 0        

        new_state[j][i] = new_state[i][j]   
    
    
    #Bell measurements

    for i in range(1,n_node-1):

        if action[i][i] == 1:  # request BM
            flag1 = 0; flag2 = 0;
            for k in range(i+1,n_node):
                if new_state[i][k] > 0:
                    node1 = k
                    node1_val = new_state[i][k]
                    flag1 = 1
                    break
            for l in range(i):
                if new_state[l][i] > 0:
                    node2 = l
                    node2_val = new_state[l][i]
                    flag2 = 1
                    break

            if flag1==1 and flag2==1:
                if np.random.random()<p_bm: #BM success
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = node1_val + node2_val - 1 #max(node1_val,node2_val) #
#                     print(new_state[node2][node1])
                    if new_state[node2][node1] > m_star:
#                         print('hellloooo')
                        new_state[node2][node1] = 0
                else:                       #BM failure
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = 0
            else:
                if flag1==1 and flag2==0:
                    new_state[i][node1] = 0
                if flag2==1 and flag1==0:
                    new_state[node2][i] = 0
                    
        for i in range(n_node):
            for j in range(i+1,n_node):
                new_state[j][i] = new_state[i][j]
                
    flagg=0
    for i in range(n_node-1):
        if action[i][i+1]>0:
            flagg=1
            break
            
    if flagg==1:
    # wait               
        for i in range(n_node-1):
            for j in range(i+1,n_node):
                if action[i][j] == 0: #wait
                    if new_state[i][j] > 0:
                        new_state[i][j] = (new_state[i][j] + 1)%(m_star+1)
                        new_state[j][i] = new_state[i][j]
                
            
    return new_state

In [135]:
# q_values = np.load("q_valuesn4m3_pauli_law_300000_pl0.5_pbm0.5.npy",allow_pickle=True)
q_values = np.load("q_valuesn4m3_wt_fid_pauli_law_300000_pbm0.5.npy",allow_pickle=True)
q_values = q_values[()].A
for state in allowed_states:
    print("state")
    print(get_state_from_index(state))
    print("action")
    print(get_next_action(get_state_from_index(state), 1))
    print("")
    print("")
    

state
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
action
[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]


state
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]]
action
[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]


state
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 2.]
 [0. 0. 2. 0.]]
action
[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]


state
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 3.]
 [0. 0. 3. 0.]]
action
[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]


state
[[0. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 0.]
 [0. 1. 0. 0.]]
action
[[0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 0.]]


state
[[0. 0. 0. 0.]
 [0. 0. 0. 2.]
 [0. 0. 0. 0.]
 [0. 2. 0. 0.]]
action
[[0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


state
[[0. 0. 0. 0.]
 [0. 0. 0. 3.]
 [0. 0. 0. 0.]
 [0. 3. 0. 0.]]
action
[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]


state
[[0. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 

In [160]:
q_values = q_values_swap_asap_4
print(get_next_action([[0., 0., 0., 0.],
 [0., 0., 3., 0],
 [0., 3., 0., 2.],
 [0., 0., 2., 0.]], 1))

[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 0.]]
